# Notebook 1 — Exploration et prise en main de Spark

## Q1 — Création de la SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from time import perf_counter
import pandas as pd

spark = (
    SparkSession.builder
    .appName("TradeCorp ETL")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Version de Spark :", spark.version)
print("Nom de l'application :", spark.sparkContext.appName)

## Q2 — Lecture des huit fichiers CSV

In [ ]:
DATA_PATH = "/home/jovyan/data"

def lire_csv(nom_fichier):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{DATA_PATH}/{nom_fichier}")
    )

df_customers = lire_csv("customers.csv")
df_orders = lire_csv("orders.csv")
df_order_details = lire_csv("order_details.csv")
df_products = lire_csv("products.csv")
df_categories = lire_csv("categories.csv")
df_suppliers = lire_csv("suppliers.csv")
df_employees = lire_csv("employees.csv")
df_shippers = lire_csv("shippers.csv")

dataframes = {
    "customers": df_customers,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employees": df_employees,
    "shippers": df_shippers,
}

print("Nombre de DataFrames chargés :", len(dataframes))

## Q3 — Exploration des schémas

In [ ]:
for nom, df in dataframes.items():
    print(f"\n===== SCHÉMA : {nom} =====")
    df.printSchema()

## Q4 — Affichage des cinq premières lignes

In [ ]:
for nom, df in dataframes.items():
    print(f"\n===== DONNÉES : {nom} =====")
    df.show(5, truncate=False)

## Q5 — Nombre de lignes par DataFrame

In [ ]:
comptages = []

for nom, df in dataframes.items():
    nombre = df.count()
    comptages.append((nom, nombre))

df_comptages = spark.createDataFrame(
    comptages,
    ["table", "nombre_lignes"]
)

df_comptages.orderBy("table").show(truncate=False)

### Observation

Les nombres de lignes correspondent au brief, sauf pour `shippers.csv`.
Le brief annonce 3 transporteurs, mais le fichier fourni contient réellement
6 lignes. Je conserve les données du fichier source sans les modifier.

## Q6 — Statistiques descriptives

In [ ]:
df_orders.select("freight").summary(
    "min",
    "max",
    "mean",
    "stddev"
).show()

In [ ]:
df_products.select(
    "unit_price",
    "units_in_stock",
    "units_on_order",
    "reorder_level"
).summary(
    "min",
    "max",
    "mean",
    "stddev"
).show()

## Q7 — Lazy evaluation

Spark utilise la **lazy evaluation**, ou évaluation paresseuse. Lorsqu'une
transformation est écrite, Spark ne traite pas immédiatement les données.
Il construit d'abord un plan d'exécution logique.

Le calcul est réellement déclenché lorsqu'une **action** est exécutée.

### Transformations

Une transformation crée un nouveau DataFrame sans lancer immédiatement le
calcul.

Exemples :

- `select()`
- `filter()`
- `withColumn()`

### Actions

Une action déclenche l'exécution du plan Spark et produit un résultat.

Exemples :

- `count()`
- `show()`
- `collect()`

Cette approche permet à Spark d'optimiser le plan d'exécution avant de
parcourir les données.

## Q8 — Consultation de Spark UI

In [ ]:
(
    df_orders
    .groupBy("ship_country")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

### Observation Spark UI

- Un **job** correspond à un traitement déclenché par une action.
- Un **stage** représente un ensemble d'opérations pouvant être exécutées
  ensemble.
- Une **task** est une unité de travail exécutée sur une partition de données.

## Q9 — Comparaison Spark et Pandas

In [ ]:
fichier_orders = f"{DATA_PATH}/orders.csv"

debut_pandas = perf_counter()

pandas_orders = pd.read_csv(fichier_orders)
nombre_pandas = len(pandas_orders)

temps_pandas = perf_counter() - debut_pandas


debut_spark = perf_counter()

spark_orders_test = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(fichier_orders)
)

# count() est indispensable car Spark utilise la lazy evaluation.
nombre_spark = spark_orders_test.count()

temps_spark = perf_counter() - debut_spark


print("Nombre de lignes Pandas :", nombre_pandas)
print("Temps Pandas :", round(temps_pandas, 6), "seconde(s)")
print("Nombre de lignes Spark :", nombre_spark)
print("Temps Spark :", round(temps_spark, 6), "seconde(s)")

### Comparaison

Sur ce petit fichier de 830 lignes, Pandas est généralement plus rapide,
car Spark doit initialiser son moteur et construire un plan d'exécution.

Pandas convient aux données de petite ou moyenne taille qui tiennent dans la
mémoire d'une seule machine.

Spark devient plus pertinent lorsque les données sont très volumineuses,
doivent être distribuées sur plusieurs machines ou nécessitent un pipeline
parallélisé et tolérant aux pannes.

## Q10 — Liste des colonnes et types

In [ ]:
print("Colonnes de df_orders :")

for colonne in df_orders.columns:
    print("-", colonne)

In [ ]:
print("Types de df_orders :")

for nom_colonne, type_colonne in df_orders.dtypes:
    print(f"{nom_colonne} : {type_colonne}")

### Colonnes qui nécessitent un cast

- `order_date` doit être convertie en date.
- `required_date` doit être convertie en date.
- `shipped_date` doit être convertie en date.
- `order_id`, `employee_id` et `ship_via` doivent être vérifiés comme entiers.
- `freight` doit être vérifiée comme valeur numérique de type double.

Les conversions seront réalisées dans le notebook de nettoyage.